# Turkmen ASR + Summarization Pipeline

Run **Block 0** first, then load models (**2A / 2B**), then run any of the four test blocks independently.

In [ ]:
!pip install -q transformers peft accelerate datasets evaluate jiwer sacrebleu soundfile librosa rouge-score bert-score

## Block 0 — Configuration

In [ ]:
# ── ASR model ─────────────────────────────────────────────────────────────────
ASR_MODEL_DIR  = "/content/drive/MyDrive/Final_models/turk_asr/output/model_35epochs"
ASR_BASE_MODEL = "facebook/mms-1b-all"
ASR_BASE_LANG  = "tur"

# ── ASR dataset source (TSV + .wav clips) ─────────────────────────────────────
ASR_CLIPS_DIR  = "/content/drive/MyDrive/Final_models/turk_asr/tk/clips"
ASR_TSV_PATH   = "/content/drive/MyDrive/Final_models/turk_asr/tk/test.tsv"
ASR_TSV_COL_PATH     = "path"      # column: audio filename
ASR_TSV_COL_SENTENCE = "sentence"  # column: reference transcription

# ── Dataset selection: "random" N samples or "manual" by index ────────────────
ASR_SELECTION_MODE  = "random"      # "random" | "manual"
ASR_RANDOM_N        = 5
ASR_MANUAL_INDICES  = [0, 1, 2, 3, 4]
ASR_RANDOM_SEED     = 42            # None = new seed each run

# ── Summarization model ────────────────────────────────────────────────────────
SUM_MODEL_DIR  = "/content/drive/MyDrive/Final_models/turk_sum/models/mbart_finetune/final"
SUM_BASE_MODEL = "facebook/mbart-large-50"
SUM_SRC_LANG   = "tr_TR"
SUM_TGT_LANG   = "tr_TR"

# ── Generation params ─────────────────────────────────────────────────────────
SUM_MAX_INPUT_LENGTH  = 512
SUM_MAX_TARGET_LENGTH = 128
SUM_NUM_BEAMS         = 4

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = "/content/drive/MyDrive/turkmen_pipeline_results"

# ── Reference summaries for pipeline (optional, keyed by subset position) ─────
# Example: {0: "Short summary for first sample..."}
SUM_REFS_PIPELINE = {}

# ── Text samples for Summarization-only block ─────────────────────────────────
SUM_TEST_SAMPLES = [
    {
        "source": (
            "Ashgabat shaherinin merkezinde yerleshyan milli kitaphana her gun "
            "munlerche okyjyny kabul edyar. Kitaphanada durli ugurlar boyuncha "
            "milliondan gowrak kitap saklanyyar. Yashlar ylym owrenmek uchin "
            "bu yere koplenchi gelyar."
        ),
        "reference": "Milli kitaphana her gun munlerche okyjyny kabul edyar we milliondan gowrak kitap saklayyar.",
    },
    {
        "source": (
            "Turkmenistanyn hokumeti taze tehnologiialary osdurmek uchin uly "
            "maya goyumlaryny amala ashyryar. Elektron hokumet ulgamy rayatlara "
            "dowlet hyzmatlaryny ansat almaga mumkinchilik beryar. "
            "Bu bashlangychlar yurdun sanly ykdysadyyetini guchlendiryar."
        ),
        "reference": "Turkmenistan taze tehnologiialara maya goyup, sanly ykdysadyyeti oshduryar.",
    },
]

## Block 1 — Imports & Utilities

In [ ]:
import os
import re
import json
import random
import torch
import torch.nn as nn
import soundfile as sf
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime


def _install_if_missing(pkg):
    import importlib, subprocess, sys
    mod = pkg.replace("-", "_").split("[")[0]
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for _pkg in ["transformers", "peft", "jiwer", "evaluate",
             "rouge_score", "bert_score", "soundfile", "librosa", "sacrebleu"]:
    _install_if_missing(_pkg)

from transformers import (
    Wav2Vec2Processor, Wav2Vec2ForCTC,
    MBart50TokenizerFast, MBartForConditionalGeneration,
)
from peft import PeftModel
from jiwer import wer as calc_wer, cer as calc_cer
import evaluate as hf_evaluate
from rouge_score import rouge_scorer
from bert_score import score as bertscore_fn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

TURKMEN_LETTERS = "abcdefghijklmnopqrstuvwxyzçäžňöşüý"

def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(f"[^{TURKMEN_LETTERS} ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

_bleu_metric = hf_evaluate.load("sacrebleu")
_chrf_metric = hf_evaluate.load("chrf")

def compute_asr_metrics(reference: str, prediction: str) -> dict:
    return {
        "WER":   calc_wer(reference, prediction),
        "CER":   calc_cer(reference, prediction),
        "BLEU":  _bleu_metric.compute(predictions=[prediction], references=[[reference]])["score"],
        "chrF2": _chrf_metric.compute(predictions=[prediction], references=[[reference]])["score"],
    }

def compute_sum_metrics(predictions: list, references: list) -> dict:
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        s = scorer.score(ref, pred)
        r1.append(s["rouge1"].fmeasure)
        r2.append(s["rouge2"].fmeasure)
        rl.append(s["rougeL"].fmeasure)
    _, _, bs = bertscore_fn(predictions, references, lang="tr",
                            rescale_with_baseline=False, verbose=False)
    return {
        "ROUGE-1":   float(np.mean(r1)),
        "ROUGE-2":   float(np.mean(r2)),
        "ROUGE-L":   float(np.mean(rl)),
        "BERTScore": float(bs.mean().item()),
    }

def print_asr_metrics(m: dict, indent: int = 5):
    p = " " * indent
    print(f"{p}WER  : {m['WER']:.4f}  ({m['WER']*100:.2f}%)")
    print(f"{p}CER  : {m['CER']:.4f}  ({m['CER']*100:.2f}%)")
    print(f"{p}BLEU : {m['BLEU']:.2f}")
    print(f"{p}chrF2: {m['chrF2']:.2f}")

def print_sum_metrics(m: dict, indent: int = 5):
    p = " " * indent
    print(f"{p}ROUGE-1  : {m['ROUGE-1']:.4f}")
    print(f"{p}ROUGE-2  : {m['ROUGE-2']:.4f}")
    print(f"{p}ROUGE-L  : {m['ROUGE-L']:.4f}")
    print(f"{p}BERTScore: {m['BERTScore']:.4f}")

def print_avg_asr(all_m: list, label: str = "", indent: int = 3):
    if not all_m:
        return
    p = " " * indent
    tag = f" ({label})" if label else ""
    print(f"\n{p}Average ASR metrics{tag} — {len(all_m)} samples:")
    for k in ["WER", "CER", "BLEU", "chrF2"]:
        vals = [m[k] for m in all_m]
        pct  = k in ("WER", "CER")
        val  = np.mean(vals) * (100 if pct else 1)
        unit = "%" if pct else ""
        print(f"{p}  {k:6s}: {val:.2f}{unit}")

def save_json(obj, filename: str):
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print(f"  Saved: {path}")


def _load_dataset() -> pd.DataFrame:
    """Load TSV dataset, keep only rows where a matching .wav file exists."""
    if not os.path.exists(ASR_TSV_PATH):
        raise FileNotFoundError(f"TSV not found: {ASR_TSV_PATH}")
    df = pd.read_csv(ASR_TSV_PATH, sep="\t")
    records = []
    for _, row in df.iterrows():
        stem     = Path(str(row[ASR_TSV_COL_PATH]).strip()).stem
        wav_path = os.path.join(ASR_CLIPS_DIR, stem + ".wav")
        if not os.path.exists(wav_path):
            continue
        records.append({"wav_path": wav_path, "reference": str(row[ASR_TSV_COL_SENTENCE]).strip()})
    return pd.DataFrame(records)

try:
    _DATASET = _load_dataset()
    print(f"Dataset loaded: {len(_DATASET)} .wav records found")
except FileNotFoundError as _e:
    _DATASET = pd.DataFrame(columns=["wav_path", "reference"])
    print(f"Warning — dataset not found: {_e}")


def get_dataset_selection() -> pd.DataFrame:
    """Return a subset from _DATASET according to Block 0 settings."""
    if _DATASET.empty:
        print("  Dataset is empty or not found.")
        return _DATASET
    if ASR_SELECTION_MODE == "manual":
        valid_idx = [i for i in ASR_MANUAL_INDICES if i < len(_DATASET)]
        subset    = _DATASET.iloc[valid_idx].reset_index(drop=True)
        print(f"  Manual selection: indices {valid_idx} -> {len(subset)} records")
    else:
        seed   = ASR_RANDOM_SEED if ASR_RANDOM_SEED is not None else random.randint(0, 99999)
        n      = min(ASR_RANDOM_N, len(_DATASET))
        subset = _DATASET.sample(n=n, random_state=seed).reset_index(drop=True)
        print(f"  Random selection: seed={seed}, n={n}")
    return subset


print(f"Block 1 ready. Device: {DEVICE}")

## Block 2A — Load ASR Model

In [ ]:
def _has_asr_checkpoint(d: str) -> bool:
    return all(Path(d, f).exists() for f in ["tokenizer_config.json", "vocab.json"])

def _load_processor_safe(d: str):
    """
    Load Wav2Vec2Processor, working around the duplicate feature_extractor conflict
    that occurs when both preprocessor_config.json and processor_config.json exist.
    """
    from transformers import Wav2Vec2FeatureExtractor, AutoTokenizer
    try:
        processor = Wav2Vec2Processor.from_pretrained(d)
        return processor
    except TypeError as e:
        if "multiple values for argument" not in str(e):
            raise
        print("  Conflict: preprocessor_config + processor_config. Building processor manually...")

    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(d)
    tokenizer = AutoTokenizer.from_pretrained(d)
    return Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

def load_asr_model():
    print("Loading ASR model...")
    d = ASR_MODEL_DIR

    if not _has_asr_checkpoint(d):
        print(f"  Checkpoint not found at {d}. Downloading base model {ASR_BASE_MODEL}...")
        Path(d).mkdir(parents=True, exist_ok=True)
        _p = Wav2Vec2Processor.from_pretrained(ASR_BASE_MODEL, target_lang=ASR_BASE_LANG)
        _p.save_pretrained(d)
        _m = Wav2Vec2ForCTC.from_pretrained(ASR_BASE_MODEL, target_lang=ASR_BASE_LANG,
                                             ignore_mismatched_sizes=True)
        _m.save_pretrained(d)
        del _p, _m; torch.cuda.empty_cache()
        print(f"  Base model saved to {d}.")
        print("  Place LoRA adapter (adapter_config.json, adapter_model.*) and lm_head.pt there, then re-run.")

    processor = _load_processor_safe(d)
    new_vocab  = len(processor.tokenizer)
    print(f"  Vocab size: {new_vocab}")

    base = Wav2Vec2ForCTC.from_pretrained(
        ASR_BASE_MODEL, target_lang=ASR_BASE_LANG,
        ignore_mismatched_sizes=True, torch_dtype=torch.float16,
    ).to(DEVICE)
    base_vocab = base.config.vocab_size

    lm_path = os.path.join(d, "lm_head.pt")
    if os.path.exists(lm_path):
        state       = torch.load(lm_path, map_location=DEVICE)
        saved_vocab = state["vocab_size"]
        hidden      = base.lm_head.in_features
        new_lm      = nn.Linear(hidden, saved_vocab, bias=True).to(DEVICE, dtype=torch.float16)
        new_lm.weight.data = state["weight"].to(DEVICE, dtype=torch.float16)
        if state["bias"] is not None:
            new_lm.bias.data = state["bias"].to(DEVICE, dtype=torch.float16)
        base.lm_head = new_lm; base.config.vocab_size = saved_vocab
        print(f"  lm_head: {base_vocab} -> {saved_vocab}")
    else:
        print("  lm_head.pt not found — expanding vocab from tokenizer...")
        if new_vocab != base_vocab:
            old    = base.lm_head
            new_lm = nn.Linear(old.in_features, new_vocab, bias=True).to(DEVICE, dtype=torch.float16)
            with torch.no_grad():
                new_lm.weight.data[:base_vocab] = old.weight.data
                if old.bias is not None:
                    new_lm.bias.data[:base_vocab] = old.bias.data
            base.lm_head = new_lm; base.config.vocab_size = new_vocab
            print(f"  lm_head: {base_vocab} -> {new_vocab}")

    model = PeftModel.from_pretrained(base, d, is_trainable=False)
    model = model.merge_and_unload()
    model.eval()
    print("ASR model ready.")
    return model, processor

ASR_MODEL, ASR_PROCESSOR = load_asr_model()

## Block 2B — Load Summarization Model

In [ ]:
def _has_sum_checkpoint(d: str) -> bool:
    return all(Path(d, f).exists() for f in ["tokenizer_config.json", "config.json"])

def load_sum_model():
    print("Loading Summarization model...")
    d = SUM_MODEL_DIR

    if not _has_sum_checkpoint(d):
        print(f"  Checkpoint not found at {d}. Downloading base model {SUM_BASE_MODEL}...")
        Path(d).mkdir(parents=True, exist_ok=True)
        _t = MBart50TokenizerFast.from_pretrained(SUM_BASE_MODEL)
        _t.save_pretrained(d)
        _m = MBartForConditionalGeneration.from_pretrained(SUM_BASE_MODEL)
        _m.save_pretrained(d)
        del _t, _m; torch.cuda.empty_cache()
        print(f"  Base model saved. Replace weights with fine-tuned and re-run.")

    tokenizer          = MBart50TokenizerFast.from_pretrained(d)
    tokenizer.src_lang = SUM_SRC_LANG
    model              = MBartForConditionalGeneration.from_pretrained(d).to(DEVICE)
    model.eval()
    print("Summarization model ready.")
    return model, tokenizer

SUM_MODEL, SUM_TOKENIZER = load_sum_model()

## Core Inference Functions

In [ ]:
def transcribe_audio(audio_path: str) -> str:
    """Transcribe a single .wav file using the fine-tuned MMS/Wav2Vec2 model."""
    audio, sr = sf.read(audio_path)
    if sr != 16000:
        import librosa
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    inp = ASR_PROCESSOR(audio, sampling_rate=16000, return_tensors="pt")
    iv  = inp.input_values.to(DEVICE, dtype=ASR_MODEL.dtype)
    with torch.no_grad():
        logits = ASR_MODEL(iv).logits
    ids = torch.argmax(logits, dim=-1)
    return ASR_PROCESSOR.batch_decode(ids, group_tokens=True)[0]

def summarize_text(text: str) -> str:
    """Summarize text using the fine-tuned mBART-50 model."""
    inp = SUM_TOKENIZER(
        text, max_length=SUM_MAX_INPUT_LENGTH,
        truncation=True, return_tensors="pt",
    ).to(DEVICE)
    with torch.no_grad():
        out = SUM_MODEL.generate(
            **inp,
            max_length=SUM_MAX_TARGET_LENGTH,
            num_beams=SUM_NUM_BEAMS,
            early_stopping=True,
            no_repeat_ngram_size=3,
            forced_bos_token_id=SUM_TOKENIZER.lang_code_to_id[SUM_TGT_LANG],
        )
    return SUM_TOKENIZER.decode(out[0], skip_special_tokens=True).strip()

## Block 1 — Full Pipeline (ASR -> Summarization)

In [ ]:
def run_pipeline():
    """Run full ASR + Summarization pipeline on dataset samples."""
    print("Block 1 — Pipeline: ASR -> Summarization")
    subset = get_dataset_selection()
    if subset.empty:
        print("Dataset is empty. Check paths in Block 0.")
        return

    asr_m_all, sum_preds, sum_refs, log = [], [], [], []

    for pos, (_, row) in enumerate(subset.iterrows()):
        path    = row["wav_path"]
        ref_raw = row["reference"]
        print(f"\n  Sample {pos+1}/{len(subset)} — {os.path.basename(path)}")

        pred_raw = transcribe_audio(path)
        print(f"  Reference : {ref_raw}")
        print(f"  Prediction: {pred_raw}")
        m_asr = compute_asr_metrics(ref_raw, pred_raw)
        print("  ASR metrics:")
        print_asr_metrics(m_asr, indent=4)
        asr_m_all.append(m_asr)

        summary = summarize_text(pred_raw)
        print(f"  Summary   : {summary}")

        entry = {
            "file": os.path.basename(path),
            "asr_reference": ref_raw, "asr_prediction": pred_raw,
            "summary": summary, "asr_metrics": m_asr,
        }

        if pos in SUM_REFS_PIPELINE:
            sum_ref = SUM_REFS_PIPELINE[pos]
            m_sum   = compute_sum_metrics([summary], [sum_ref])
            print(f"  Summary ref: {sum_ref}")
            print("  Summary metrics:")
            print_sum_metrics(m_sum, indent=4)
            entry.update({"sum_reference": sum_ref, "sum_metrics": m_sum})
            sum_preds.append(summary); sum_refs.append(sum_ref)

        log.append(entry)

    print_avg_asr(asr_m_all, label="pipeline")
    if sum_preds:
        agg = compute_sum_metrics(sum_preds, sum_refs)
        print(f"\n  Average Summary metrics ({len(sum_preds)} samples):")
        print_sum_metrics(agg)
    save_json(log, "pipeline_results.json")
    print("\nDone.")

run_pipeline()

## Block 2 — ASR Only

In [ ]:
def run_asr_only():
    """Run ASR transcription only, with metrics against reference transcriptions."""
    print("Block 2 — ASR Only")
    subset = get_dataset_selection()
    if subset.empty:
        print("Dataset is empty. Check paths in Block 0.")
        return

    m_raw_all, m_norm_all, log = [], [], []

    for i, row in subset.iterrows():
        path    = row["wav_path"]
        ref_raw = row["reference"]
        print(f"\n  Sample {i+1}/{len(subset)} — {os.path.basename(path)}")

        pred_raw  = transcribe_audio(path)
        ref_norm  = normalize_text(ref_raw)
        pred_norm = normalize_text(pred_raw)
        m_raw     = compute_asr_metrics(ref_raw, pred_raw)
        m_norm    = compute_asr_metrics(ref_norm, pred_norm)
        m_raw_all.append(m_raw)
        m_norm_all.append(m_norm)

        print(f"  Reference : {ref_raw}")
        print(f"  Prediction: {pred_raw}")
        print("  Metrics (raw):")
        print_asr_metrics(m_raw)
        print("  Metrics (normalized):")
        print_asr_metrics(m_norm)

        log.append({
            "file": os.path.basename(path),
            "reference_raw":  ref_raw,  "prediction_raw":  pred_raw,
            "reference_norm": ref_norm, "prediction_norm": pred_norm,
            "metrics_raw": m_raw,       "metrics_norm": m_norm,
        })

    print_avg_asr(m_raw_all,  label="raw")
    print_avg_asr(m_norm_all, label="normalized")
    save_json(log, "asr_only_results.json")
    print("\nDone.")

run_asr_only()

## Block 3 — Summarization Only

In [ ]:
def run_sum_only():
    """Run summarization on SUM_TEST_SAMPLES defined in Block 0."""
    print("Block 3 — Summarization Only")
    preds, refs, log = [], [], []

    for i, sample in enumerate(SUM_TEST_SAMPLES, 1):
        src  = sample["source"]
        ref  = sample["reference"]
        pred = summarize_text(src)
        preds.append(pred); refs.append(ref)

        print(f"\n  Sample {i}")
        print(f"  Source    : {src[:110]}{'...' if len(src) > 110 else ''}")
        print(f"  Reference : {ref}")
        print(f"  Prediction: {pred}")
        m = compute_sum_metrics([pred], [ref])
        print("  Metrics:")
        print_sum_metrics(m)
        log.append({"sample": i, "source": src, "reference": ref, "prediction": pred, "metrics": m})

    if len(preds) > 1:
        agg = compute_sum_metrics(preds, refs)
        print(f"\n  Average metrics ({len(preds)} samples):")
        print_sum_metrics(agg)
        log.append({"aggregate_metrics": agg})

    save_json(log, "sum_only_results.json")
    print("\nDone.")

run_sum_only()

## Block 4 — Custom Input

Upload your own audio files or provide raw text. Set `CUSTOM_MODE` and configure the appropriate inputs below.

In [ ]:
# ── Custom input configuration ────────────────────────────────────────────────
#
#  CUSTOM_MODE options:
#   "pipeline"  — audio file(s) -> ASR -> Summarization
#   "asr"       — audio file(s) -> ASR only
#   "sum"       — raw text      -> Summarization only
#
CUSTOM_MODE = "pipeline"   # "pipeline" | "asr" | "sum"

# ── For modes "pipeline" and "asr": list of audio file paths ──────────────────
# Google Drive example: "/content/drive/MyDrive/my_audio.wav"
# Colab upload example: "/content/my_audio.wav"
CUSTOM_AUDIO_FILES = [
    # "/content/drive/MyDrive/audio_test/my_audio_1.wav",
    # "/content/drive/MyDrive/audio_test/my_audio_2.wav",
]

# Optional reference transcriptions, keyed by list index (0-based)
CUSTOM_ASR_REFERENCES = {
    # 0: "Expected transcription for first file...",
}

# ── For mode "sum": list of text strings ─────────────────────────────────────
CUSTOM_TEXTS = [
    # "Paste your Turkmen text here...",
]

# Optional reference summaries, keyed by list index (0-based)
CUSTOM_SUM_REFERENCES = {
    # 0: "Expected summary for first text...",
}

# ── Optionally load audio paths from a dataset TSV (overrides CUSTOM_AUDIO_FILES)
# Set to True to load from the dataset configured in Block 0 instead
USE_DATASET_FOR_CUSTOM = False


def run_custom():
    print(f"Block 4 — Custom Input  [mode: {CUSTOM_MODE}]")

    # Resolve audio file list
    if CUSTOM_MODE in ("pipeline", "asr"):
        if USE_DATASET_FOR_CUSTOM:
            subset = get_dataset_selection()
            audio_list = list(zip(
                subset["wav_path"].tolist(),
                subset["reference"].tolist()
            ))
        else:
            audio_list = [
                (p, CUSTOM_ASR_REFERENCES.get(i, ""))
                for i, p in enumerate(CUSTOM_AUDIO_FILES)
            ]
        if not audio_list:
            print("  No audio files configured. Add paths to CUSTOM_AUDIO_FILES or set USE_DATASET_FOR_CUSTOM=True.")
            return

    log = []

    if CUSTOM_MODE == "sum":
        if not CUSTOM_TEXTS:
            print("  No texts configured. Add strings to CUSTOM_TEXTS.")
            return
        for i, text in enumerate(CUSTOM_TEXTS):
            pred = summarize_text(text)
            print(f"\n  Text {i+1}: {text[:110]}{'...' if len(text) > 110 else ''}")
            print(f"  Summary : {pred}")
            entry = {"text": text, "summary": pred}
            if i in CUSTOM_SUM_REFERENCES:
                ref   = CUSTOM_SUM_REFERENCES[i]
                m     = compute_sum_metrics([pred], [ref])
                print(f"  Reference: {ref}")
                print("  Metrics:")
                print_sum_metrics(m)
                entry.update({"reference": ref, "metrics": m})
            log.append(entry)

    else:
        asr_m_all = []
        for i, (path, ref_raw) in enumerate(audio_list):
            if not os.path.exists(path):
                print(f"  File not found: {path}")
                continue
            print(f"\n  Audio {i+1}/{len(audio_list)} — {os.path.basename(path)}")
            pred_raw = transcribe_audio(path)
            print(f"  Reference : {ref_raw if ref_raw else '(none)'}")
            print(f"  Prediction: {pred_raw}")
            entry = {"file": os.path.basename(path), "prediction": pred_raw}
            if ref_raw:
                m_asr = compute_asr_metrics(ref_raw, pred_raw)
                print("  ASR metrics:")
                print_asr_metrics(m_asr)
                asr_m_all.append(m_asr)
                entry["reference"] = ref_raw
                entry["asr_metrics"] = m_asr

            if CUSTOM_MODE == "pipeline":
                summary = summarize_text(pred_raw)
                print(f"  Summary   : {summary}")
                entry["summary"] = summary
                if i in CUSTOM_SUM_REFERENCES:
                    ref_sum = CUSTOM_SUM_REFERENCES[i]
                    m_sum   = compute_sum_metrics([summary], [ref_sum])
                    print(f"  Sum ref   : {ref_sum}")
                    print("  Sum metrics:")
                    print_sum_metrics(m_sum)
                    entry.update({"sum_reference": ref_sum, "sum_metrics": m_sum})
            log.append(entry)

        if asr_m_all:
            print_avg_asr(asr_m_all)

    save_json(log, "custom_results.json")
    print("\nDone.")

run_custom()